In [0]:
import hashlib
import json
import urllib.request
import warnings

from datetime import datetime, timezone
from pathlib import Path
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [0]:
def download_feed(name, url, snapshot_dir, run_id):
    """Save original feed and metadata in a unique run directory."""
    path = Path(snapshot_dir) / f"{name}.txt"
    metadata_path = Path(snapshot_dir) / f"{name}.metadata.json"

    metadata = {
        "run_id": run_id,
        "feed_name": name,
        "source_url": url,
        "snapshot_path": str(path),
        "download_started_at": datetime.now(timezone.utc).isoformat(),
        "download_finished_at": None,
        "size_bytes": None,
        "sha256": None,
        "download_status": "failed",
        "error": None,
    }
    
    try:
        # Save the original file without parsing or changing its content.
        urllib.request.urlretrieve(url, str(path))
        content = path.read_bytes()

        metadata.update({
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
            "download_status": "downloaded",
        })

    except Exception as e:
        metadata["error"] = f"{type(e).__name__}: {e}"

        # remove a potentially incomplete download
        if path.exists():
            path.unlink()

        raise RuntimeError(
            f"Error downloading {name} ({url}): {e}"
        ) from e

    finally:
        metadata["download_finished_at"] = (
            datetime.now(timezone.utc).isoformat()
        )
        metadata_path.write_text(
            json.dumps(metadata, indent=2),
            encoding="utf-8",
        )

    print(f"Downloaded {name} -> {path}")
    return str(path)

In [0]:
def parse_domains(df: DataFrame, rule: str, return_counts=False):
    """
    Extract hostnames from hosts or simple adblock rules and classify
    them as 'accepted domain', 'IP', or 'wrong format'.

    Skips system hostnames and lines outside the supported input format.
    Preserves subdomains and duplicate records.
    Supports ASCII domain names and IPv4 addresses.

    df: expects raw_line, feed_name, and ingestion_ts columns.
    rule: either 'hosts' or 'adblock'
    With return_counts=True, also return input/skipped/alias/system-host counts.
    Returns a Spark DataFrame with columns: raw_line, domain, validation_status, feed_name, ingestion_ts   
    """

    system_hosts = [
        "localhost", "localhost.localdomain", "local", "broadcasthost",
        "ip6-localhost", "ip6-loopback", "ip6-localnet",
        "ip6-mcastprefix", "ip6-allnodes", "ip6-allrouters", "ip6-allhosts",
    ]

    label = r"[a-z0-9_](?:[a-z0-9_-]{0,61}[a-z0-9_])?"
    domain_pattern = rf"^{label}(?:\.{label})+$"

    # IPv4: 4 numbers between 0–255 divided with dots
    octet = r"(?:25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])"
    ipv4_pattern = rf"^{octet}(?:\.{octet}){{3}}$"

    counts = {}
    if return_counts:
        counts["input lines"] = df.count()
    df = df.withColumn("line", F.lower(F.trim("raw_line")))

    if rule == "hosts":
        supported = F.trim(F.regexp_replace("line", r"#.*$", "")).rlike(
            r"^(?:0\.0\.0\.0|127\.0\.0\.1|::1)\s+"
        )
        if return_counts:
            supported_count = df.filter(supported).count()
        df = (
            # remove comments (crucial for the ones after the real domains)
            df.withColumn(
                "line",
                F.trim(F.regexp_replace("line", r"#.*$", ""))
            )
            .filter(F.col("line").rlike(r"^(?:0\.0\.0\.0|127\.0\.0\.1|::1)\s+"))
            .withColumn("tokens", F.split("line", r"\s+"))
            .withColumn("domain", F.explode(F.expr("slice(tokens, 2, size(tokens) - 1)")))
        )

    elif rule == "adblock":
        if return_counts:
            supported_count = df.filter(F.col("line").rlike(r"^\|\|.*\^$")).count()
        df = (
            #.filter(F.col("line").rlike(r"^\|\|[A-Za-z0-9.-]+\^$")) # for now we want to parse everything and clean later
            df.filter(F.col("line").rlike(r"^\|\|.*\^$"))
            .withColumn(
                "domain",
                F.regexp_extract("line", r"^\|\|(.*)\^$", 1)
            )
        )

    else:
        raise ValueError("rule should be defined as either 'hosts' or 'adblock'")
    
    # Normalize before validation, so a valid trailing DNS dot is not rejected.
    df = df.withColumn("domain", F.regexp_replace("domain", r"\.$", ""))
    if return_counts:
        counts["skipped lines"] = counts["input lines"] - supported_count
        counts["extra aliases"] = df.count() - supported_count
        counts["system hosts"] = df.filter(F.col("domain").isin(system_hosts + ["0.0.0.0"])).count()

    df = (
        df.filter(~F.col("domain").isin(system_hosts + ["0.0.0.0"]))
        .withColumn(
            "validation_status",
            F.when(
                # create a separate rule for IPv4
                F.col("domain").rlike(ipv4_pattern),
                F.lit("IP")
            ).when(
                # verify domain correctness
                F.col("domain").rlike(domain_pattern)
                & (F.length("domain") <= 253)
                # wrong IPv4 format shouldn't passs ass domain
                & ~F.col("domain").rlike(r"^\d+(?:\.\d+){3}$"),
                F.lit("accepted domain")
            ).otherwise(F.lit("wrong format"))
        )
        .select("raw_line", "domain", "validation_status", "feed_name", "ingestion_ts")
    )

    return (df, counts) if return_counts else df

In [0]:
def log_record_count(
    audit_table,
    run_id,
    run_date,
    feed_name,
    stage,
    record_count,
    status,
    previous_count=None,
    ratio=None,
):
    """Append a single audit row to the feed_count_audit table."""
    row = {
        "run_id": run_id,
        "run_date": run_date,
        "feed_name": feed_name,
        "stage": stage,
        "status": status,
        "record_count": int(record_count),
        "previous_count": (
            int(previous_count) if previous_count is not None else None
        ),
        "ratio": ratio,
        "logged_at": datetime.now(timezone.utc),
    }

    audit_schema = spark.table(audit_table).schema

    (
        spark.createDataFrame([row], schema=audit_schema)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )

In [0]:
def get_previous_parsed_count(audit_table, feed_name, run_id, stage="parsed"):
    """Return the record count from the most recent successful parse of the same feed, excluding the current run."""
    rows = (
        spark.table(audit_table)
        .filter(
            (F.col("feed_name") == feed_name)
            & (F.col("stage") == stage)
            & (F.col("status") == "passed")
            & (F.col("run_id") != run_id)
        )
        .orderBy(
            F.col("logged_at").desc(),
        )
        .select("record_count")
        .limit(1)
        .collect()
    )

    return int(rows[0]["record_count"]) if rows else None